# Step 4 – Network losses in the heat-supply model (DHNx → oemof)
**Sweden · SE3 2019 · same five producers and assumptions as Step 3 · demand + DHNx distribution losses · no storage**

Step 3 optimised the capacity and hourly dispatch of the five producers for the consumers' heat demand only. In reality the producers must also cover the heat lost from the pipes on the way to the consumers. This notebook adds the distribution losses of the Step 2 network to the demand,

$$Q_{total}(t) = Q_{demand}(t) + Q_{losses}(t),$$

and solves **the same oemof model as Step 3** again (`build_and_solve_heat_system` in `helpers.py`). Prices, technologies, costs and capacity limits are unchanged, so every difference between Step 3 and Step 4 is caused by the losses.

| | |
|---|---|
| **Reads** | `data/step3/` (`oemof_input.csv`, `technology_parameters.csv`, `carriers.csv`) · `results/step3/` (`unit_summary.csv`, `dispatch.csv`) · `results/step2/` (`network_losses.csv`, `losses_per_pipe.csv`, `heat_demand_plus_losses_hourly.csv`) |
| **Writes** | `data/step4_loss/oemof_input_with_losses.csv` · `results/step4_loss/` (tables, figures, Excel file) |

Run Step 2 and Step 3 first. Units: DHNx works in kW, oemof in MW; the conversion is done once, in section 2.

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (DATA, RESULTS, SOLVER, INTEREST_RATE,
                     build_and_solve_heat_system, summarize_units,
                     heat_dispatch, plot_dispatch, UNIT_COLORS)

STEP3_IN, STEP3_OUT = DATA / "step3", RESULTS / "step3"
IN, OUT = DATA / "step4_loss", RESULTS / "step4_loss"
IN.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Colours: each unit keeps its Step 3 colour; Step 3 vs Step 4 comparisons use grey vs dark red.
C_STEP3, C_STEP4 = "#9E9E9E", "#B2182B"
print("Solver:", SOLVER)

## 1. Step 3 inputs, unchanged
Everything except the demand is taken from the files Step 3 saved: hourly prices, the technology table (efficiencies, CAPEX, variable O&M, lifetimes, capacity limits) and the carrier table. The Step 3 results are loaded as the reference case for the comparison in section 8.

In [ ]:
for f in ["oemof_input.csv", "technology_parameters.csv", "carriers.csv"]:
    assert (STEP3_IN / f).exists(), f"Missing data/step3/{f}: run Step 3 first"
for f in ["unit_summary.csv", "dispatch.csv"]:
    assert (STEP3_OUT / f).exists(), f"Missing results/step3/{f}: run Step 3 first"

data = pd.read_csv(STEP3_IN / "oemof_input.csv", index_col=0, parse_dates=True)
data.index = pd.DatetimeIndex(data.index, freq="h")      # oemof needs a regular hourly index
tech = pd.read_csv(STEP3_IN / "technology_parameters.csv", index_col=0)
carriers = pd.read_csv(STEP3_IN / "carriers.csv", index_col=0)
summary_s3 = pd.read_csv(STEP3_OUT / "unit_summary.csv", index_col=0)
dispatch_s3 = pd.read_csv(STEP3_OUT / "dispatch.csv", index_col=0, parse_dates=True)

assert len(data) == 8760 and data.index.is_unique
assert (dispatch_s3.index == data.index).all(), "Step 3 dispatch has a different time index"
units = tech.index.tolist()
assert summary_s3.index.drop("TOTAL").tolist() == units, "Step 3 summary and technology table list different units"

print(f"Step 3 demand: peak {data['heat demand'].max():.3f} MW | annual {data['heat demand'].sum():,.1f} MWh")
display(tech[["DHNx producer", "input", "efficiency", "capex_eur_per_MW",
              "var_om_eur_per_MWh", "max_capacity_MW", "lifetime_yr"]])

## 2. Distribution losses from DHNx (Step 2)
In the Step 2 pipe table every pipe type has `l_factor = 0` and a fixed loss `l_factor_fix` [kW/m]. DHNx therefore gives each **built** pipe a loss of `l_factor_fix × length`, independent of the heat flow: the pipes are kept hot all year, so the loss is the **same in every hour**. Step 2 exported the sum over all built pipes to `results/step2/network_losses.csv`.

🔧 **ASSUMPTION** (state in the report): constant hourly network loss. In reality losses fall slightly in summer when supply temperatures and soil temperature change; this model does not capture that.

Checks before the losses are used (grade B asks for consistency of units and time resolution):
1. the total loss equals the sum of the per-pipe losses;
2. Step 2's hourly demand (kW) equals Step 3's demand (MW) × 1000 in every one of the 8760 hours, so both models use the same consumers, profiles and hours.

In [ ]:
net_losses = pd.read_csv(RESULTS / "step2/network_losses.csv")
pipe_losses = pd.read_csv(RESULTS / "step2/losses_per_pipe.csv")
s2_hourly = pd.read_csv(RESULTS / "step2/heat_demand_plus_losses_hourly.csv")

loss_kW = float(net_losses.loc[0, "total_loss_kW"])

# Check 1: total loss = sum over the built pipes
assert np.isclose(loss_kW, pipe_losses["losses"].sum(), rtol=1e-6), "Total loss differs from the sum per pipe"
# Check 2: same hours and same demand in Step 2 (kW) and Step 3 (MW)
assert len(s2_hourly) == len(data) == 8760, "Time resolution differs (expected 8760 hourly values)"
assert np.allclose(s2_hourly["demand_kW"].to_numpy() / 1000, data["heat demand"].to_numpy(), atol=1e-6), \
    "Step 2 and Step 3 demand profiles differ"

loss_MW = loss_kW / 1000                     # kW (DHNx) -> MW (oemof)
print(f"Network heat loss: {loss_kW:.2f} kW = {loss_MW:.4f} MW, constant in all 8760 h")
print(f"Annual loss: {loss_MW * 8760:,.1f} MWh/yr  (Step 2 file: {net_losses.loc[0, 'annual_loss_MWh']:,.1f} MWh/yr)")
print(f"Built pipes: {len(pipe_losses)} | built length {net_losses.loc[0, 'built_length_m']:,.0f} m")
print("✅ Units and time resolution consistent between DHNx and oemof")

print("\nPipes with the largest losses:")
display(pipe_losses.sort_values("losses", ascending=False).head(8).round(3))

## 3. Adjusted load profile: Q_total = Q_demand + Q_losses
The adjusted load is saved as a new input file, `data/step4_loss/oemof_input_with_losses.csv`, column `heat demand incl. losses` (the original `heat demand` column is kept for the comparison).

Because the loss is constant while the demand varies, the losses are a **larger share of the heat supplied in summer** than in winter.

In [ ]:
data["network losses"] = loss_MW                                        # MW, every hour
data["heat demand incl. losses"] = data["heat demand"] + data["network losses"]
data.to_csv(IN / "oemof_input_with_losses.csv")

q_dem, q_tot = data["heat demand"], data["heat demand incl. losses"]
load_table = pd.DataFrame({
    "Step 3: demand only": [q_dem.max(), q_dem.mean(), q_dem.min(), q_dem.sum()],
    "Step 4: demand + losses": [q_tot.max(), q_tot.mean(), q_tot.min(), q_tot.sum()],
}, index=["peak [MW]", "mean [MW]", "minimum [MW]", "annual [MWh]"])
load_table["change [%]"] = 100 * (load_table["Step 4: demand + losses"] / load_table["Step 3: demand only"] - 1)
display(load_table.round(3))

loss_share_year = 100 * loss_MW * 8760 / q_tot.sum()
print(f"Losses = {loss_share_year:.1f} % of the heat the producers supply over the year "
      f"({100 * loss_MW / q_tot.max():.1f} % in the peak hour, {100 * loss_MW / q_tot.min():.1f} % in the lowest-demand hour)")

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
ax = axs[0]
ax.plot(np.sort(q_tot.values)[::-1], color=C_STEP4, lw=2, label="Step 4: demand + losses")
ax.plot(np.sort(q_dem.values)[::-1], color=C_STEP3, lw=2, label="Step 3: demand only")
ax.set_xlabel("Hours of the year (sorted)"); ax.set_ylabel("Heat load [MW]")
ax.set_title("Load duration curve"); ax.legend(); ax.grid(alpha=0.3)

monthly_share = 100 * data["network losses"].resample("MS").sum() / q_tot.resample("MS").sum()
ax = axs[1]
ax.bar(monthly_share.index.strftime("%b"), monthly_share.values, color=C_STEP4, width=0.7)
ax.set_ylabel("Losses / heat supplied [%]"); ax.set_title("Share of network losses per month")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "load_with_losses.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Optimisation problem (same as Step 3)
Decision variables, objective and constraints are those of Step 3 section 6. The only change is the hourly heat balance:

$$\sum_u Q_{u,t} = D_t + L \qquad \text{for every hour } t, \qquad L = \text{network loss [MW]}$$

The capacities are **optimised again**, not fixed to the Step 3 values. This answers how the design must change to cover the losses; section 8.3 additionally checks whether the Step 3 design alone would have been sufficient.

In [ ]:
model, results, parts = build_and_solve_heat_system(
    data, tech, carriers, demand_col="heat demand incl. losses", i=INTEREST_RATE)

sr = model.solver_results
status = (sr.solver.termination_condition if hasattr(sr, "solver")          # CBC: pyomo SolverResults
          else sr.get("termination_condition", "n/a"))                   # HiGHS: plain dict
print("Solver:", SOLVER, "| termination condition:", status)
print(f"Objective (total annual cost): {model.objective():,.0f} EUR/a   "
      f"(Step 3: {summary_s3.loc['TOTAL', 'total cost [EUR]']:,.0f} EUR/a)")

## 5. Verify that demand + losses are satisfied
Hourly heat balance: the sum of all unit outputs must equal demand + losses in **every** hour.

In [ ]:
dispatch = heat_dispatch(results, parts)          # MW per unit + 'demand' (= demand + losses here)
imbalance = dispatch[units].sum(axis=1) - q_tot
print(f"Max hourly imbalance: {imbalance.abs().max():.2e} MW")
print(f"Heat produced {dispatch[units].sum().sum():,.1f} MWh vs demand + losses {q_tot.sum():,.1f} MWh")
assert imbalance.abs().max() < 1e-4, "Demand not met!"
print("✅ Demand + losses met in all 8760 hours")

## 6. Results per unit (Step 4)
Same table as Step 3 section 8. The total must equal the objective.

In [ ]:
summary = summarize_units(results, parts, tech, carriers)
assert np.isclose(model.objective(), summary.loc["TOTAL", "total cost [EUR]"], rtol=1e-6, atol=0.1), \
    "Cost report differs from objective"
for s in (summary, summary_s3):
    s["operating cost [EUR]"] = s["fuel/electricity cost [EUR]"] + s["var O&M cost [EUR]"]

display(summary[["capacity [MW]", "heat [MWh]", "share of heat [%]", "full-load hours [h]",
                 "hours at max capacity [h]", "operating cost [EUR]", "total cost [EUR]"]].round(2))
print(f"Annualised system cost per MWh delivered to consumers: "
      f"{summary.loc['TOTAL', 'total cost [EUR]'] / q_dem.sum():.2f} EUR/MWh "
      f"(Step 3: {summary_s3.loc['TOTAL', 'total cost [EUR]'] / q_dem.sum():.2f} EUR/MWh)")

## 7. Dispatch over the year (Step 4)
### 7.1 Whole year and duration curves
Stacked heat production per unit (daily means) with demand + losses as the black line, and the duration curve of each unit. ✏️ Describe which units supply base, mid and peak load and compare with Step 3 (section 9 there).

In [ ]:
plot_dispatch(dispatch, "Heat production by unit – Step 4 (demand + network losses)")
plt.savefig(OUT / "dispatch.png", dpi=150, bbox_inches="tight"); plt.show()

### 7.2 Hourly dispatch in a winter and a summer week
The solid black line is demand + losses (what the producers must supply), the dashed line the consumers' demand only. The gap between them is the network loss. It is the same in both weeks, but a much larger part of the load in summer.

In [ ]:
weeks = [("2019-01-14", "2019-01-20", "winter week"), ("2019-07-15", "2019-07-21", "summer week")]
fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharey=True)
for ax, (start, end, label) in zip(axs, weeks):
    week = dispatch.loc[start:end]
    ax.stackplot(week.index, *[week[u] for u in units], labels=units,
                 colors=[UNIT_COLORS[u] for u in units])
    ax.plot(week.index, week["demand"], "k", lw=1.3, label="demand + losses")
    ax.plot(week.index, q_dem.loc[start:end], "k--", lw=1, label="demand only")
    ax.set_title(f"Hourly dispatch, {label} – Step 4"); ax.set_ylabel("MW"); ax.grid(alpha=0.3)
axs[1].legend(ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=9, frameon=False)
plt.tight_layout(); plt.savefig(OUT / "dispatch_weeks.png", dpi=150, bbox_inches="tight"); plt.show()

### 7.3 Monthly heat production by unit

In [ ]:
monthly = dispatch[units].resample("MS").sum()          # MWh per month
monthly.index = monthly.index.strftime("%b")
ax = monthly.plot.bar(stacked=True, figsize=(12, 4), width=0.8,
                      color=[UNIT_COLORS[u] for u in units])
ax.set_ylabel("Heat produced [MWh/month]"); ax.set_xlabel("")
ax.set_title("Monthly heat production by unit – Step 4 (demand + network losses)")
ax.legend(ncol=3, loc="upper center"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(OUT / "dispatch_monthly.png", dpi=150, bbox_inches="tight"); plt.show()

### 7.4 When do units reach their maximum capacity?
Hours per month in which each unit runs at ≥ 99.9 % of its installed capacity (same definition as Step 3 section 10).

In [ ]:
caps = summary.loc[units, "capacity [MW]"]
at_cap = pd.DataFrame({u: dispatch[u] >= 0.999 * caps[u] for u in units if caps[u] > 1e-6})
hours_at_cap_by_month = at_cap.groupby(at_cap.index.month).sum()
hours_at_cap_by_month.index.name = "month"
display(hours_at_cap_by_month)

## 8. How the losses changed the design and the operation (Step 3 vs Step 4)
### 8.1 Per unit and for the whole system
⚠️ The two heat pumps have identical technology data, so the solver may split heat and capacity between them differently in each run. Only their **sum** is meaningful; the figures below therefore group them as "heat pumps (P0 + P4)".

In [ ]:
cols = ["capacity [MW]", "heat [MWh]", "share of heat [%]", "full-load hours [h]",
        "hours at max capacity [h]", "operating cost [EUR]", "total cost [EUR]", "CO2 [t]"]
comparison = pd.concat({"Step 3": summary_s3[cols], "Step 4": summary[cols]}, axis=1)
for c in cols:
    comparison[("Change", c)] = comparison[("Step 4", c)] - comparison[("Step 3", c)]
comparison = comparison[["Step 3", "Step 4", "Change"]]
display(comparison.round(3))

def system_totals(s, load, losses_MWh):
    tot = s.loc["TOTAL"]
    return pd.Series({
        "heat produced [MWh]": tot["heat [MWh]"],
        "  of which network losses [MWh]": losses_MWh,
        "peak load [MW]": load.max(),
        "installed capacity [MW]": tot["capacity [MW]"],
        "annualised CAPEX [EUR/yr]": tot["annualised capex [EUR]"],
        "fuel/electricity cost [EUR/yr]": tot["fuel/electricity cost [EUR]"],
        "variable O&M [EUR/yr]": tot["var O&M cost [EUR]"],
        "operating cost [EUR/yr]": tot["operating cost [EUR]"],
        "total annual cost [EUR/yr]": tot["total cost [EUR]"],
        "cost per MWh delivered to consumers [EUR/MWh]": tot["total cost [EUR]"] / q_dem.sum(),
        "CO2 [t/yr]": tot["CO2 [t]"],
        "delivered / produced heat [%]": 100 * q_dem.sum() / tot["heat [MWh]"],
    })

system = pd.DataFrame({"Step 3": system_totals(summary_s3, q_dem, 0.0),
                       "Step 4": system_totals(summary, q_tot, data["network losses"].sum())})
system["change"] = system["Step 4"] - system["Step 3"]
system["change [%]"] = 100 * system["change"] / system["Step 3"].where(system["Step 3"].abs() > 1e-6)   # no % for rows that are 0 in Step 3
display(system.round(3))

### 8.2 Graphical comparison
1. **Design:** installed capacity per unit in Step 3 and Step 4, with the assumed site limit (`max_capacity_MW`) marked. A unit at its limit cannot grow further, so any extra capacity has to come from another unit.
2. **Where the extra heat comes from:** change in annual heat production per unit. The bars add up to the annual network loss.
3. **When the extra heat is produced:** change in monthly heat production per unit.
4. **Costs:** annual system cost split into annualised CAPEX, fuel/electricity and variable O&M.

In [ ]:
HP_GROUP = "heat pumps (P0 + P4)"
group_of = {u: (HP_GROUP if tech.loc[u, "technology_type"] == "heat_pump" else u) for u in units}
groups = list(dict.fromkeys(group_of.values()))
GROUP_COLORS = {**UNIT_COLORS, HP_GROUP: UNIT_COLORS["heat pump"]}

def by_group(df):
    # sum the unit columns of df (hourly or monthly) into the groups
    return df[units].T.groupby(lambda u: group_of[u], sort=False).sum().T[groups]

fig, axs = plt.subplots(1, 2, figsize=(15, 4.8))

# (1) design: capacity per unit, Step 3 vs Step 4, with site limit
ax = axs[0]
y = np.arange(len(units)); h = 0.38
limit_y = tech.loc[units, "max_capacity_MW"].astype(float).tolist()
cap3 = summary_s3.loc[units, "capacity [MW]"].astype(float)
cap4 = summary.loc[units, "capacity [MW]"].astype(float)
ax.barh(y + h / 2, cap3, height=h, color=C_STEP3, label="Step 3: demand only")
ax.barh(y - h / 2, cap4, height=h, color=C_STEP4, label="Step 4: demand + losses")
ax.scatter(tech.loc[units, "max_capacity_MW"], y, marker="|", s=500, color="k", zorder=3, label="site limit")
for yi, (c3, c4) in enumerate(zip(cap3, cap4)):
    ax.text(max(c3, c4, limit_y[yi]) + 0.006, yi, f"{1000 * (c4 - c3):+.1f} kW", va="center", fontsize=9)
ax.set_yticks(y, units); ax.invert_yaxis()
ax.set_xlabel("Installed heat capacity [MW]"); ax.set_title("Design: installed capacity per unit")
ax.set_xlim(0, tech["max_capacity_MW"].max() * 1.25)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.13), fontsize=9, frameon=False)
ax.grid(axis="x", alpha=0.3)

# (2) change in annual heat per unit group
ax = axs[1]
heat3 = by_group(dispatch_s3).sum(); heat4 = by_group(dispatch).sum()
dheat = heat4 - heat3
ax.bar(groups, dheat, color=[GROUP_COLORS[g] for g in groups], width=0.6)
for i, v in enumerate(dheat):
    ax.text(i, v, f"{v:+.1f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=9)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Change in annual heat [MWh/yr]")
ax.set_title(f"Where the extra {dheat.sum():.0f} MWh/yr come from (= annual network loss)")
ax.tick_params(axis="x", labelsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "comparison_design_heat.png", dpi=150, bbox_inches="tight"); plt.show()

# (3) change in monthly heat per unit group
m3 = by_group(dispatch_s3.resample("MS").sum()); m4 = by_group(dispatch.resample("MS").sum())
dmonth = m4 - m3
dmonth.index = dmonth.index.strftime("%b")
ax = dmonth.plot.bar(stacked=True, figsize=(12, 4), width=0.8, color=[GROUP_COLORS[g] for g in groups])
ax.plot(range(12), dmonth.sum(axis=1).values, "ko-", ms=4, lw=1, label="total (= monthly network loss)")
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Change in heat [MWh/month]"); ax.set_xlabel("")
ax.set_title("Change in monthly heat production, Step 4 minus Step 3")
ax.set_ylim(min(0, dmonth.clip(upper=0).sum(axis=1).min()) * 1.2, dmonth.clip(lower=0).sum(axis=1).max() * 1.12)
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.1), fontsize=9, frameon=False)
plt.xticks(rotation=0); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "comparison_monthly_change.png", dpi=150, bbox_inches="tight"); plt.show()

# (4) annual cost split
cost_parts = ["annualised capex [EUR]", "fuel/electricity cost [EUR]", "var O&M cost [EUR]"]
costs = pd.DataFrame({"Step 3: demand only": summary_s3.loc["TOTAL", cost_parts],
                      "Step 4: demand + losses": summary.loc["TOTAL", cost_parts]}).T.astype(float)
costs.columns = ["annualised CAPEX", "fuel/electricity", "variable O&M"]
ax = costs.plot.barh(stacked=True, figsize=(12, 3.2), width=0.6, color=["#5B7FA6", "#E3A33B", "#8BB174"])
for i, total in enumerate(costs.sum(axis=1)):
    ax.text(total + 500, i, f"{total:,.0f} EUR/yr", va="center", fontsize=9)
ax.set_xlabel("Annual system cost [EUR/yr]"); ax.set_title("Annual system cost, Step 3 vs Step 4")
ax.set_xlim(0, costs.sum(axis=1).max() * 1.18); ax.invert_yaxis()
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.28), fontsize=9, frameon=False); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "comparison_costs.png", dpi=150, bbox_inches="tight"); plt.show()

### 8.3 Do units now operate closer to their capacity limits?
Two views:
- **Re-optimised design (Step 4):** how much headroom is left between each unit's capacity and its site limit, and how many hours it runs at full capacity compared with Step 3.
- **Step 3 design kept unchanged:** if the Step 3 capacities were built and the losses only discovered afterwards, in how many hours could demand + losses not be supplied?

In [ ]:
limit = tech.loc[units, "max_capacity_MW"].astype(float)
headroom = pd.DataFrame({
    "site limit [MW]": limit,
    "capacity Step 3 [MW]": cap3,
    "capacity Step 4 [MW]": cap4,
    "use of limit Step 3 [%]": 100 * cap3 / limit,
    "use of limit Step 4 [%]": 100 * cap4 / limit,
    "hours at max Step 3 [h]": summary_s3.loc[units, "hours at max capacity [h]"].astype(float),
    "hours at max Step 4 [h]": summary.loc[units, "hours at max capacity [h]"].astype(float),
})
headroom.loc["TOTAL"] = [limit.sum(), cap3.sum(), cap4.sum(), 100 * cap3.sum() / limit.sum(),
                         100 * cap4.sum() / limit.sum(), np.nan, np.nan]
display(headroom.round(3))

cap_s3_total = cap3.sum()
shortfall = (q_tot - cap_s3_total).clip(lower=0)
n_short = int((shortfall > 1e-6).sum())
print(f"Step 3 installed capacity: {cap_s3_total:.3f} MW | new peak load (demand + losses): {q_tot.max():.3f} MW")
print(f"With the Step 3 design, demand + losses exceed the installed capacity in {n_short} hours "
      f"(largest shortfall {1000 * shortfall.max():.1f} kW, {shortfall.sum():.2f} MWh/yr unserved).")
if n_short:
    print("Months with a shortfall:", sorted(set(shortfall[shortfall > 1e-6].index.strftime("%b"))))

## 9. Save results
CSV files for later steps and one Excel file with all tables for the report. Figures were saved above.

In [ ]:
summary.to_csv(OUT / "unit_summary.csv")
dispatch.to_csv(OUT / "dispatch.csv")
hours_at_cap_by_month.to_csv(OUT / "hours_at_capacity_by_month.csv")
comparison.to_csv(OUT / "comparison_step3_step4_units.csv")
system.to_csv(OUT / "comparison_step3_step4_system.csv")
headroom.to_csv(OUT / "capacity_headroom.csv")

excel_file = OUT / "step4_loss_results.xlsx"
try:
    with pd.ExcelWriter(excel_file) as writer:
        system.round(3).to_excel(writer, sheet_name="System Step3 vs Step4")
        comparison.round(3).to_excel(writer, sheet_name="Units Step3 vs Step4")
        headroom.round(3).to_excel(writer, sheet_name="Capacity headroom")
        summary.round(3).to_excel(writer, sheet_name="Unit summary Step4")
        hours_at_cap_by_month.to_excel(writer, sheet_name="Hours at capacity")
        dispatch.resample("MS").sum().round(3).to_excel(writer, sheet_name="Monthly heat (MWh)")
        dispatch.round(4).to_excel(writer, sheet_name="Hourly dispatch (MW)")
        load_table.round(3).to_excel(writer, sheet_name="Load with losses")
        data.to_excel(writer, sheet_name="Inputs (hourly)")
    print("Saved", excel_file)
except ImportError:
    print("Excel file skipped: run  %pip install openpyxl  once, then rerun this cell.")
print("Saved CSV files and figures to", OUT)

## 10. Key numbers for the answers
Printed from the results above, so the answers in section 11 can be checked against the current run.

In [ ]:
hp = [u for u in units if group_of[u] == HP_GROUP]
print(f"Network loss: {loss_kW:.1f} kW constant = {loss_MW * 8760:,.0f} MWh/yr "
      f"= {loss_share_year:.1f} % of the heat produced")
print(f"Heat produced: {system.loc['heat produced [MWh]', 'Step 3']:,.0f} -> {system.loc['heat produced [MWh]', 'Step 4']:,.0f} MWh/yr")
print(f"Installed capacity: {cap3.sum():.3f} -> {cap4.sum():.3f} MW")
print(f"Operating cost: {system.loc['operating cost [EUR/yr]', 'Step 3']:,.0f} -> {system.loc['operating cost [EUR/yr]', 'Step 4']:,.0f} EUR/yr "
      f"({system.loc['operating cost [EUR/yr]', 'change [%]']:+.1f} %)")
print(f"Total cost: {system.loc['total annual cost [EUR/yr]', 'Step 3']:,.0f} -> {system.loc['total annual cost [EUR/yr]', 'Step 4']:,.0f} EUR/yr "
      f"({system.loc['total annual cost [EUR/yr]', 'change [%]']:+.1f} %)")
print(f"Cost per MWh delivered: {system.loc['cost per MWh delivered to consumers [EUR/MWh]', 'Step 3']:.2f} -> "
      f"{system.loc['cost per MWh delivered to consumers [EUR/MWh]', 'Step 4']:.2f} EUR/MWh")
print("\nShare of heat [%] per group, Step 3 -> Step 4:")
share3 = 100 * heat3 / heat3.sum(); share4 = 100 * heat4 / heat4.sum()
for g in groups:
    print(f"  {g:24s} {share3[g]:6.2f} -> {share4[g]:6.2f}   (heat {heat3[g]:8.1f} -> {heat4[g]:8.1f} MWh, {dheat[g]:+.1f})")

## 11. Questions to answer
✏️ Write the answers with the numbers from section 10.

1. **How does including distribution losses change the operation of each generating unit?** *(sections 7, 8.1, 8.2: change in annual and monthly heat per unit, dispatch weeks)*

2. **Do any units now operate closer to their capacity limits?** *(section 8.3: use of the site limit, hours at maximum capacity, and the shortfall of the Step 3 design)*

3. **How are total heat generation, operating cost and the generation mix affected?** *(sections 3 and 8.1: heat produced, operating and total cost, cost per MWh delivered, share of heat per unit)*

Hints for the discussion (grade B):
- Show explicitly how the DHNx loss was transferred: kW from `network_losses.csv` → MW, added as a constant to every hour (section 2–3), and the two consistency checks.
- The loss is constant, so it matters most in summer (section 3): which unit supplies it then, and which in winter?
- Explain any change in capacity with the merit order: the loss raises the peak load by the same amount as every other hour, so the cheapest unit with headroom below its site limit takes the extra capacity.
- The two heat pumps are identical: discuss them together, not one by one.